# Notebook 04 — Baseline Inference

> **阶段**：Stage 2 Experiment · **预计时间**：30–90 分钟（取决于模式）· **平台**：Kaggle Notebook（GPU）

| 资源 | fast | teaching | research |
| --- | --- | --- | --- |
| 页数 | 12 | 100 | 1651 |
| 实测时间 | ≈ 3 h（CPU，894 s/页） | CPU 不可行（≈25 h） | 仅 GPU 或分段（见 README 排障） |
| 磁盘 | 小 | 中 | 需缓存策略 |


# Learning Objectives

完成本 Notebook 后，你应该能够：

- 建立 SmolDocling 的 **zero-shot baseline**（不做任何训练/调 prompt）；
- 使用分层抽样得到可解释的评测子集；
- 运行带缓存、断点恢复的批量推理；
- 解释为什么 baseline 是后续所有对比的锚点。


# Why This Matters

没有 baseline，任何 SFT/LoRA 的提升都无法归因。Baseline 回答：「这个模型在固定数据、固定 prompt、固定采样下，zero-shot 是什么水平？」之后每次实验只改变一个变量。


# Concepts

- **三种模式**：`fast`（调试）/ `teaching`（教学评测）/ `research`（官方 1651 页全量，仅评测）；
- **分层抽样**：按文档类型轮转抽样，避免某一类型主导小样本；
- **缓存与恢复**：每个样本一个 JSON，重启后自动跳过已有输出——Notebook 关闭不丢结果；
- **元数据**：每个预测记录 prompt_id / model revision / image_id / generation_config / latency。


## Step 1 — 配置与模式

所有默认值在 `configs/default.yaml`。Baseline 使用官方默认 prompt `v0`，`do_sample=False`（贪心解码），保证可复现。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src.config import load_config

cfg = load_config()
print('modes:', cfg['modes'])
print('generation:', cfg['generation'])


## Step 2 — 加载模型并选择子集


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data
from src.model import SmolDoclingAdapter, model_summary

MODE = 'fast'  # TODO: 学生改为 teaching 观察规模差异

data_root = data.find_dataset_root()
annotations = data.load_annotations(data_root)

adapter = SmolDoclingAdapter().load()
print(model_summary(adapter))


## Step 3 — 批量推理（缓存 + 进度条 + 断点恢复）

产物结构（`results/baseline/`）：

```text
predictions/<image_id>.json   # 每次推理的完整记录
doctags/<image_id>.dt         # 原始 DocTags
manifest.jsonl                # 本次所有样本行
summary.json                  # 延迟/数量汇总
```


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data
from src.config import project_root
from src.inference import run_baseline

output_dir = project_root() / cfg['paths']['baseline_dir']
manifest = run_baseline(
    annotations,
    data_root,
    adapter,
    output_dir=output_dir,
    mode=MODE,
    config=cfg,
    prompt_id='v0',
    skip_existing=True,
)
print('manifest:', manifest)


## Step 4 — 检查结果与元数据


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

import json
import collections
from src import data

rows = [json.loads(line) for line in manifest.read_text(encoding='utf-8').splitlines() if line.strip()]
summary = data.read_json(manifest.parent / 'summary.json')
print('summary:', json.dumps(summary, ensure_ascii=False, indent=2))
doc_types = collections.Counter(r['document_type'] for r in rows)
print('文档类型分布:', dict(doc_types))
print('样例 doctags（前 500 字符）:')
print(rows[0]['doctags'][:500])
print('样例元数据字段:', sorted(k for k in rows[0] if k != 'doctags'))


## Step 5 — 验证缓存与断点恢复

再次运行同一命令：全部样本应从缓存命中（skipped = 全部），耗时应接近 0。这就是「Notebook 重启后不丢结果」的机制。


In [ ]:

# 仓库路径定位：兼容「Notebook 位于仓库根目录」与「仓库克隆在 /kaggle/working 子目录」
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "model.py").exists():
    matches = [p for p in REPO_ROOT.iterdir() if p.is_dir() and (p / "src" / "model.py").exists()]
    if not matches:
        raise FileNotFoundError(
            "未找到仓库根目录。请按 notebooks/README.md 把仓库克隆到 /kaggle/working，"
            "或把本 Notebook 放在仓库根目录。"
        )
    REPO_ROOT = matches[0].resolve()
sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)

from src import data
from src.config import project_root
from src.inference import run_baseline

manifest2 = run_baseline(
    annotations, data_root, adapter,
    output_dir=project_root() / cfg['paths']['baseline_dir'],
    mode=MODE, config=cfg, prompt_id='v0', skip_existing=True,
)
summary2 = data.read_json(manifest2.parent / 'summary.json')
print('第二次运行 skipped_from_cache =', summary2['skipped_from_cache'], '/', summary2['completed'])


# What You Should Observe

- summary.json 里的 mean_latency_sec 是单页推理成本——对比研究模式的预算；
- 抽样子集按文档类型分层，小样本下分布更均衡；
- 每条 prediction 都携带 model_revision 与 prompt_id，证据链完整。


# Research Checkpoint

> **为什么 baseline 必须是「固定数据 + 固定 prompt + 固定采样」？** 如果中途换了任何一项，后续 SFT 前后对比还能成立吗？

**TODO：** 把答案写在 `results/nb04/research_checkpoint.md`。


# Exercises

1. **TODO：** 把 `MODE` 改为 `teaching` 重跑，记录耗时与磁盘增量，估算 research 模式预算；
2. **TODO：** 换 `prompt_id='v1'` 跑一个 fast 子集（注意会新建缓存条目），粗看 doctags 与 v0 的差异；
3. **TODO：** 修改 `configs/default.yaml` 的 `generation.do_sample=true` 后重跑 3 页，观察输出随机性与 latency 变化，然后改回 false。


# Takeaways

- Baseline 是所有实验的锚点；
- 缓存/断点恢复让 Kaggle 会话限制不再致命；
- 元数据 = 证据链，缺失元数据的分数没有科研价值。

**下一步**：[Notebook 07](07_Benchmark_and_Evaluation.ipynb) — 用官方评测给 baseline 打分。
